<a href="https://colab.research.google.com/github/juanseazurmendi-ui/Full-control-historial/blob/que-psasa2/models/colab/design_template_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FullControl design template

*<<< check out demo models [here](https://github.com/FullControlXYZ/fullcontrol/tree/master/models/README.md) >>>*
  
press ctrl+F9 to run all cells in this notebook, or press shift+enter to run each cell sequentially

if you change one of the code cells, make sure you run it and all subsequent cells again (in order)

*this document is a jupyter notebook - if they're new to you, check out how they work: [link](https://www.google.com/search?q=ipynb+tutorial), [link](https://jupyter.org/try-jupyter/retro/notebooks/?path=notebooks/Intro.ipynb), [link](https://colab.research.google.com/)*
### be patient :)

the next code cell may take a while because running it causes several things to happen:
- connect to a google colab server -> download the fullcontrol code -> install the fullcontrol code

check out [other tutorials](https://github.com/FullControlXYZ/fullcontrol/blob/master/tutorials/README.md) to understand the python code for the FullControl design

In [43]:
if 'google.colab' in str(get_ipython()):
  !pip install git+https://github.com/FullControlXYZ/fullcontrol --quiet
import fullcontrol as fc
import numpy as np
from google.colab import files

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [44]:
# printer/gcode parameters

design_name = 'my_design'
nozzle_temp = 210
bed_temp = 40
print_speed = 1000
fan_percent = 100
printer_name='ender_3' # generic / ultimaker2plus / prusa_i3 / ender_3 / cr_10 / bambulab_x1 / toolchanger_T0

In [45]:
# design parameters

EW = 0.4 # extrusion width
EH = 0.2 # extrusion height (and layer height)
initial_z = EH*0.6 # initial nozzle position is set to 0.6x the extrusion height to get a bit of 'squish' for good bed adhesion
layers = 50

In [46]:

# ============================================================
# CONFIGURACIÓN DE PARÁMETROS
# ============================================================

# Dimensiones de la canasta (en mm)
ancho_total = 50  # 5 cm
alto_total = 50   # 5 cm

# Centro de la pieza
centro_x = 50
centro_y = 50

# Dimensiones de la canasta
radio_base = 12        # Radio en la base (2.4 cm de diámetro)
radio_maximo = 20      # Radio máximo a mitad de altura (4 cm de diámetro)
radio_top = 18         # Radio en la apertura (3.6 cm de diámetro)
altura_canasta = 40    # Altura de la canasta (4 cm)

# Parámetros de impresión
altura_capa = 0.2      # Altura de cada capa en mm
ancho_extrusion = 0.4  # Ancho de la línea de extrusión

# Base de sujeción
capas_base = 1         # Capas sólidas en la base
radio_base_sujecion = radio_base + 3  # Radio de la base de sujeción

# Parámetros del patrón de tejido
vueltas_canasta = 35   # Número de vueltas de la espiral
segmentos_por_vuelta = 64  # Segmentos por vuelta (más = más suave)
amplitud_ondulacion = 1.5  # Amplitud del patrón de tejido en mm

# Velocidades (mm/min)
velocidad_primera_capa = 600
velocidad_normal = 1200
velocidad_retraccion = 1800

# Temperatura (ajustar según tu material)
temp_extrusor = 200  # Para PLA estándar
temp_cama = 60

# ============================================================
# INICIALIZACIÓN
# ============================================================

steps = []

# Configuración inicial del extrusor
steps.append(fc.GcodeComment(text='=== CANASTA 3D - Generado con FullControl ==='))
steps.append(fc.GcodeComment(text=f'Dimensiones: {ancho_total}x{ancho_total}x{alto_total} mm'))

# Configurar geometría de extrusión
steps.append(fc.ExtrusionGeometry(
    width=ancho_extrusion,
    height=altura_capa
))

# ============================================================
# CREAR BASE DE SUJECIÓN (ESPIRAL SÓLIDA)
# ============================================================

steps.append(fc.GcodeComment(text='--- BASE DE SUJECIÓN ---'))
steps.append(fc.Printer(print_speed=velocidad_primera_capa))

for capa in range(capas_base):
    z_actual = altura_capa * (capa + 1)

    # Espiral desde el centro hacia afuera para crear base sólida
    espirales_base = 40  # Número de vueltas en la espiral de la base

    for i in range(espirales_base * segmentos_por_vuelta):
        progreso_espiral = i / (espirales_base * segmentos_por_vuelta)

        # Radio crece desde 0 hasta radio_base_sujecion
        radio_actual = radio_base_sujecion * progreso_espiral

        # Ángulo de la espiral
        angulo = progreso_espiral * espirales_base * tau

        x = centro_x + radio_actual * cos(angulo)
        y = centro_y + radio_actual * sin(angulo)

        steps.append(fc.Point(x=x, y=y, z=z_actual))

    # Completar el perímetro final de la base en esta capa
    for seg in range(segmentos_por_vuelta + 1):
        angulo = (seg / segmentos_por_vuelta) * tau
        x = centro_x + radio_base_sujecion * cos(angulo)
        y = centro_y + radio_base_sujecion * sin(angulo)
        steps.append(fc.Point(x=x, y=y, z=z_actual))

# ============================================================
# TRANSICIÓN DE BASE A CANASTA
# ============================================================

steps.append(fc.GcodeComment(text='--- TRANSICIÓN BASE-CANASTA ---'))

# Crear anillo de transición que conecta base sólida con inicio de canasta
z_transicion = altura_capa * (capas_base + 1)

for seg in range(segmentos_por_vuelta):
    progreso_seg = seg / segmentos_por_vuelta
    angulo = progreso_seg * tau

    # Radio transiciona de radio_base_sujecion a radio_base
    radio_transicion = radio_base_sujecion + (radio_base - radio_base_sujecion) * progreso_seg

    x = centro_x + radio_transicion * cos(angulo)
    y = centro_y + radio_transicion * sin(angulo)
    steps.append(fc.Point(x=x, y=y, z=z_transicion))

# ============================================================
# CANASTA CON PATRÓN DE TEJIDO
# ============================================================

steps.append(fc.GcodeComment(text='--- CUERPO DE LA CANASTA ---'))
steps.append(fc.Printer(print_speed=velocidad_normal))

z_inicio_canasta = altura_capa * (capas_base + 1)

for vuelta in range(vueltas_canasta):
    for seg in range(segmentos_por_vuelta):
        # Progreso total de la canasta (0 a 1)
        progreso = (vuelta * segmentos_por_vuelta + seg) / (vueltas_canasta * segmentos_por_vuelta)

        # Función de radio base (forma de canasta)
        if progreso < 0.4:
            # Primera parte: expande desde base hasta máximo
            t = progreso / 0.4
            radio_base_actual = radio_base + (radio_maximo - radio_base) * t
        else:
            # Segunda parte: contrae suavemente hacia la apertura
            t = (progreso - 0.4) / 0.6
            radio_base_actual = radio_maximo + (radio_top - radio_maximo) * t

        # Agregar patrón ondulado para simular tejido
        angulo = (seg / segmentos_por_vuelta) * tau

        # Patrón de tejido: combina ondulaciones verticales y horizontales
        ondulacion_vertical = amplitud_ondulacion * sin(vuelta * tau / 3)
        ondulacion_horizontal = amplitud_ondulacion * sin(angulo * 8)
        ondulacion = ondulacion_vertical + ondulacion_horizontal * 0.5

        radio_actual = radio_base_actual + ondulacion

        # Calcular posición
        x = centro_x + radio_actual * cos(angulo)
        y = centro_y + radio_actual * sin(angulo)
        z = z_inicio_canasta + progreso * altura_canasta

        steps.append(fc.Point(x=x, y=y, z=z))

# ============================================================
# BORDE SUPERIOR (REFUERZO)
# ============================================================

steps.append(fc.GcodeComment(text='--- BORDE SUPERIOR ---'))

z_final = z_inicio_canasta + altura_canasta

# Agregar 2-3 vueltas extra en el borde superior para reforzar
for vuelta_borde in range(3):
    for seg in range(segmentos_por_vuelta):
        angulo = (seg / segmentos_por_vuelta) * tau
        x = centro_x + radio_top * cos(angulo)
        y = centro_y + radio_top * sin(angulo)
        z = z_final + vuelta_borde * altura_capa
        steps.append(fc.Point(x=x, y=y, z=z))

# ============================================================
# FINALIZACIÓN
# ============================================================

steps.append(fc.GcodeComment(text='--- FIN DE IMPRESIÓN ---'))

# ============================================================
# GENERAR GCODE
# ============================================================

# Configuración de la impresora (ajustar según tu modelo)
gcode_controls = fc.GcodeControls(
    printer_name='ender_3',  # Cambiar según tu impresora
    save_as='canasta_5x5x5',
    initialization_data={
        'primer': 'front_lines_then_y',
        'print_speed': velocidad_normal,
        'nozzle_temp': temp_extrusor,
        'bed_temp': temp_cama,
        'fan_percent': 100
    }
)

# Generar el archivo GCode
print("Generando GCode...")
gcode = fc.transform(steps, 'gcode', gcode_controls)
print("¡GCode generado exitosamente!")
print(f"Archivo guardado como: canasta_5x5x5.gcode")
print(f"\nEstadísticas de la pieza:")
print(f"- Dimensiones: ~{ancho_total}x{ancho_total}x{alto_total + capas_base * altura_capa:.1f} mm")
print(f"- Altura total: {z_final + 3 * altura_capa:.1f} mm")
print(f"- Capas base: {capas_base}")
print(f"- Vueltas canasta: {vueltas_canasta}")
print(f"- Puntos totales: {len(steps)}")

# ============================================================
# VISUALIZACIÓN (OPCIONAL)
# ============================================================

# Descomentar para ver preview antes de imprimir
# print("\nGenerando vista previa...")
fc.transform(steps, 'plot', fc.PlotControls(style='tube'))

Generando GCode...
fc.transform guidance tips are being written to screen if any potential issues are found - hide tips with fc.transform(..., show_tips=False)
tip: set initial `extrusion_width` and `extrusion_height` in the initialization_data to ensure the correct amount of material is extruded:
   - `fc.transform(..., controls=fc.GcodeControls(initialization_data={'extrusion_width': EW, 'extrusion_height': EH}))`

¡GCode generado exitosamente!
Archivo guardado como: canasta_5x5x5.gcode

Estadísticas de la pieza:
- Dimensiones: ~50x50x50.8 mm
- Altura total: 41.6 mm
- Capas base: 4
- Vueltas canasta: 35
- Puntos totales: 13006
fc.transform guidance tips are being written to screen if any potential issues are found - hide tips with fc.transform(..., show_tips=False)
tip: set initial `extrusion_width` and `extrusion_height` in the initialization_data to ensure the preview is correct:
   - `fc.transform(..., controls=fc.PlotControls(initialization_data={'extrusion_width': EW, 'extrusion

In [47]:
# preview the design

fc.transform(steps, 'plot', fc.PlotControls(style='line', zoom=0.7))
# hover the cursor over the lines in the plot to check xyz positions of the points in the design

# uncomment the next line to create a plot with real heights/widths for extruded lines to preview the real 3D printed geometry
# fc.transform(steps, 'plot', fc.PlotControls(style='tube', zoom=0.7, initialization_data={'extrusion_width': EW, 'extrusion_height': EH}))

# uncomment the next line to create a neat preview (click the top-left button in the plot for a .png file) - post and tag @FullControlXYZ :)
# fc.transform(steps, 'plot', fc.PlotControls(neat_for_publishing=True, zoom=0.5, initialization_data={'extrusion_width': EW, 'extrusion_height': EH}))


In [48]:
# generate and save gcode

gcode_controls = fc.GcodeControls(
    printer_name=printer_name,

    initialization_data={
        'primer': 'front_lines_then_y',
        'print_speed': print_speed,
        'nozzle_temp': nozzle_temp,
        'bed_temp': bed_temp,
        'fan_percent': fan_percent,
        'extrusion_width': EW,
        'extrusion_height': EH})
gcode = fc.transform(steps, 'gcode', gcode_controls)
open(f'{design_name}.gcode', 'w').write(gcode)
files.download(f'{design_name}.gcode')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#### please tell us what you're doing with FullControl!

- tag FullControlXYZ on social media ([twitter](https://twitter.com/FullControlXYZ), [instagram](https://www.instagram.com/fullcontrolxyz/), [linkedin](https://www.linkedin.com/in/andrew-gleadall-068587119/), [tiktok](https://www.tiktok.com/@fullcontrolxyz))
- email [info@fullcontrol.xyz](mailto:info@fullcontrol.xyz)
- post on the [subreddit](https://reddit.com/r/fullcontrol)
- post in the [github discussions or issues tabs](https://github.com/FullControlXYZ/fullcontrol/issues)

in publications, please cite the original FullControl paper and the github repo for the new python version:

- Gleadall, A. (2021). FullControl GCode Designer: open-source software for unconstrained design in additive manufacturing. Additive Manufacturing, 46, 102109.
- Gleadall, A. and Leas, D. (2023). FullControl [electronic resource: python source code]. available at: https://github.com/FullControlXYZ/fullcontrol